# Clase en vivo 2 · Machine Learning aplicado

**Caso:** NovaRetail · **Pregunta de la clase:** ¿qué modelo y qué umbral debería usar NovaRetail para priorizar contactos sin saturar al equipo comercial?

Cómo se usa este notebook: corré las celdas **en orden**. Antes de cada una hay una explicación breve de qué hace y qué mirar. La teoría la vemos en la presentación; acá se aplica.

Todos los datos son **sintéticos**: se generan desde el código y siempre dan lo mismo. Ningún resultado representa una situación real de ARCA.

## 0 · Preparación: los datos de NovaRetail

Esta celda carga las librerías y define el generador de clientes de NovaRetail. Es el mismo generador de toda la cursada: misma semilla, mismos clientes. Solo hace falta correrla.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

VIOLETA, VERDE, NARANJA, GRIS, ROJO = "#7C5CF6", "#10D9A0", "#F59E0B", "#8888A8", "#E5484D"
plt.rcParams.update({"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 11, "axes.titleweight": "bold"})
pd.options.display.float_format = "{:,.3f}".format

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.calibration import calibration_curve

COLUMNAS = ["id_cliente", "fecha_alta", "canal", "segmento", "region", "edad", "antiguedad_meses", "compras_12m",
            "ticket_promedio", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app", "descuento_promedio",
            "satisfaccion", "motivo_baja", "abandono_90d"]


def generar_novaretail(n=8000, semilla=2025, duplicados=60):
    rng = np.random.default_rng(semilla)
    fecha_corte = pd.Timestamp("2025-12-31")
    canal = rng.choice(["web", "app", "tienda"], size=n, p=[0.40, 0.25, 0.35])
    # composición: web concentra más clientes del segmento Esencial (esto confunde la lectura por canal)
    p_seg = {"web": [0.62, 0.28, 0.10], "app": [0.30, 0.45, 0.25], "tienda": [0.28, 0.42, 0.30]}
    segmento = np.array([rng.choice(["Esencial", "Plus", "Premium"], p=p_seg[c]) for c in canal])
    region = rng.choice(["Centro", "Norte", "Sur", "Litoral", "Cuyo"], size=n, p=[0.30, 0.20, 0.18, 0.17, 0.15])
    edad = np.clip(rng.normal(41, 12, n), 18, 80).round().astype(int)
    alta = pd.Timestamp("2021-01-01") + pd.to_timedelta(rng.integers(0, 1730, n), unit="D")
    antig = ((fecha_corte - alta).days / 30.4).round().astype(int)
    base_compras = np.select([segmento == "Esencial", segmento == "Plus"], [5, 9], 14)
    compras = rng.poisson(base_compras)
    ticket = np.round(np.select([segmento == "Esencial", segmento == "Plus"], [35, 62], 110) * rng.lognormal(0, 0.25, n), 2)
    dias = np.clip(rng.gamma(2.2, 24, n), 1, 365).round().astype(int)
    soporte = rng.poisson(np.where(segmento == "Esencial", 1.2, 0.6))
    usa_app = np.where(canal == "app", 1, (rng.random(n) < 0.35).astype(int))
    descuento = np.clip(rng.normal(np.where(segmento == "Esencial", 0.14, 0.08), 0.05, n), 0, 0.35).round(3)
    sat_real = np.clip(np.round(rng.normal(3.7 - 0.12 * soporte + 0.15 * (segmento == "Premium"), 0.9, n)), 1, 5)

    z_dias = (dias - 60) / 40
    logit = (-2.45
             + 0.85 * (segmento == "Esencial") - 0.45 * (segmento == "Premium")
             + 0.80 * z_dias + 0.30 * soporte - 0.42 * (sat_real - 3.7)
             - 0.50 * usa_app - 0.012 * (antig - 24) - 0.04 * (compras - 8)
             + 0.55 * ((canal == "tienda") & (segmento == "Esencial")))
    p = 1 / (1 + np.exp(-logit))
    abandono = (rng.random(n) < p).astype(int)

    # faltantes: la satisfacción falta más entre quienes abandonan (faltante informativo)
    sat = sat_real.copy()
    sat[rng.random(n) < np.where(abandono == 1, 0.30, 0.09)] = np.nan
    ticket = ticket.astype(float)
    ticket[rng.random(n) < 0.03] = np.nan
    region_s = pd.Series(region, dtype=object)
    region_s[rng.random(n) < 0.02] = np.nan
    # valores imposibles
    edad = edad.astype(float)
    malos = rng.random(n) < 0.004
    edad[malos] = rng.choice([0, 150, -1], size=malos.sum())
    # fuga: el motivo de baja solo existe para quienes ya abandonaron
    motivo = np.where(abandono == 1, rng.choice(["precio", "servicio", "competencia", "mudanza"], size=n), None)

    df = pd.DataFrame({
        "id_cliente": [f"NR{i:06d}" for i in range(1, n + 1)],
        "fecha_alta": alta.strftime("%Y-%m-%d"),
        "canal": canal, "segmento": segmento, "region": region_s,
        "edad": edad, "antiguedad_meses": antig, "compras_12m": compras,
        "ticket_promedio": ticket, "dias_desde_ultima_compra": dias,
        "contactos_soporte_90d": soporte, "usa_app": usa_app,
        "descuento_promedio": descuento, "satisfaccion": sat,
        "motivo_baja": motivo, "abandono_90d": abandono,
    })
    # duplicados: filas repetidas por un error de carga
    if duplicados:
        df = pd.concat([df, df.sample(duplicados, random_state=semilla)], ignore_index=True)
    return df.sample(frac=1, random_state=semilla).reset_index(drop=True)

Ahora generamos la tabla y aplicamos la limpieza que ya hicimos en la clase 1, en tres líneas:

1. Nos quedamos con **una fila por cliente** (el generador agrega 60 filas repetidas por un «error de carga»).
2. Las edades imposibles (menores de 18 o mayores de 100) pasan a faltante.
3. Creamos un **indicador de satisfacción faltante**: en NovaRetail, que falte la encuesta de satisfacción dice algo sobre el cliente.

In [ ]:
clientes = generar_novaretail()                       # 8.060 filas: 8.000 clientes + 60 repetidas
df = clientes.drop_duplicates("id_cliente").reset_index(drop=True)
df.loc[(df["edad"] < 18) | (df["edad"] > 100), "edad"] = np.nan
df["satisfaccion_faltante"] = df["satisfaccion"].isna().astype(int)

print(f"Filas originales: {len(clientes):,} · Clientes únicos: {len(df):,}")
df.head()

## 1 · Tipo de problema

Antes de elegir un modelo, hay que decir **qué estamos prediciendo**. En NovaRetail, la columna `abandono_90d` vale 1 si el cliente abandonó en los 90 días siguientes a la fecha de corte (31/12/2025) y 0 si siguió comprando.

**Predicción antes de correr:** ¿es un problema de regresión, de clasificación o de clustering? ¿Qué tasa de abandono esperás?

In [ ]:
conteo = df["abandono_90d"].value_counts().sort_index()
tasa_base = df["abandono_90d"].mean()

fig, ax = plt.subplots()
ax.bar(["Siguió (0)", "Abandonó (1)"], conteo.values, color=[VIOLETA, NARANJA])
for i, v in enumerate(conteo.values):
    ax.text(i, v, f"{v:,}\n({v / len(df):.1%})", ha="center", va="bottom")
ax.set_title("Es un problema de clasificación binaria: abandona o no")
ax.set_ylabel("Clientes"); ax.set_ylim(0, conteo.max() * 1.18)
plt.show()
print(f"Tasa base de abandono: {tasa_base:.1%}")

Lo que se ve: la variable a predecir tiene **dos categorías**, así que es **clasificación binaria**. No es regresión (no predecimos un monto) ni clustering (sí conocemos la respuesta de los clientes pasados). Y está **desbalanceada**: abandona uno de cada siete clientes, alrededor del 14 %. Esa tasa base va a importar cuando miremos el accuracy.

## 2 · Partición estratificada

Separamos los clientes en **entrenamiento** (75 %) y **prueba** (25 %). Con `stratify` cuidamos que las dos partes tengan la misma tasa de abandono, y con `random_state` que la partición sea siempre la misma: así todos los modelos se comparan sobre las mismas filas.

**Predicción antes de correr:** ¿qué tasa de abandono va a tener el conjunto de prueba?

In [ ]:
NUM = ["edad", "antiguedad_meses", "compras_12m", "ticket_promedio", "dias_desde_ultima_compra",
       "contactos_soporte_90d", "usa_app", "descuento_promedio", "satisfaccion", "satisfaccion_faltante"]
CAT = ["canal", "segmento", "region"]

X, y = df[NUM + CAT], df["abandono_90d"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

print(f"Entrenamiento: {len(X_train):,} clientes · tasa de abandono {y_train.mean():.2%}")
print(f"Prueba:        {len(X_test):,} clientes · tasa de abandono {y_test.mean():.2%}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(["Entrenamiento", "Prueba"], [y_train.mean(), y_test.mean()], color=[VIOLETA, VERDE])
ax.set_ylim(0, 0.2); ax.set_title("Misma tasa de abandono en las dos partes"); ax.set_ylabel("Tasa de abandono")
plt.show()

## 3 · Qué variables entran

No todas las columnas se pueden usar. La regla: **solo variables que existan en el momento de tomar la decisión**. Esta tabla revisa, una por una, qué entra y qué no.

In [ ]:
variables = pd.DataFrame({
    "variable": ["canal", "segmento", "region", "edad", "antiguedad_meses", "compras_12m", "ticket_promedio",
                 "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app", "descuento_promedio",
                 "satisfaccion", "motivo_baja", "id_cliente", "fecha_alta"],
    "tipo": ["categórica"] * 3 + ["numérica"] * 9 + ["categórica", "identificador", "fecha"],
})
variables["faltantes"] = [df[v].isna().mean() for v in variables["variable"]]
variables["entra"] = ~variables["variable"].isin(["motivo_baja", "id_cliente", "fecha_alta"])
variables["motivo"] = ["Disponible al decidir"] * 12 + ["Solo existe DESPUÉS de abandonar: es fuga",
                       "Identificador: no describe al cliente", "Ya está resumida en antiguedad_meses"]
variables.style.format({"faltantes": "{:.1%}"}).hide(axis="index")

Cómo leer la tabla:

- `motivo_baja` tiene 86 % de faltantes y **no entra**: solo se completa para quienes ya abandonaron. Usarla sería mirar la respuesta antes de predecir. Eso se llama **fuga** y lo medimos en la sección 13.
- `id_cliente` y `fecha_alta` no entran: uno identifica y la otra ya está resumida en `antiguedad_meses`.
- `satisfaccion` falta en un 13 %. No la borramos: imputamos y **además** dejamos el indicador `satisfaccion_faltante`, porque que falte es información.

## 4 · Pipeline con preprocesamiento

Un **Pipeline** junta, en un solo objeto, el preprocesamiento y el modelo. Lo importante: se **ajusta solo con entrenamiento**. La mediana que usa para imputar y la escala que usa para estandarizar se calculan con `X_train` y después se aplican igual a `X_test`. Así la prueba no se cuela en el entrenamiento.

El primer modelo es una **regresión logística**: combina las variables con pesos y devuelve una **probabilidad** de abandono entre 0 y 1.

In [ ]:
def preprocesamiento():
    return ColumnTransformer([
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]), NUM),
        ("cat", Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                          ("codificar", OneHotEncoder(handle_unknown="ignore"))]), CAT),
    ])

def armar_pipeline(modelo):
    return Pipeline([("preprocesamiento", preprocesamiento()), ("modelo", modelo)])

regresion_logistica = armar_pipeline(LogisticRegression(max_iter=1000)).fit(X_train, y_train)
p_rl = regresion_logistica.predict_proba(X_test)[:, 1]

print("Pasos del pipeline:", [nombre for nombre, _ in regresion_logistica.steps])
print(f"AUC en prueba: {roc_auc_score(y_test, p_rl):.3f}")
pd.DataFrame({"probabilidad_de_abandono": p_rl[:5], "abandono_real": y_test.values[:5]})

## 5 · El accuracy engaña

El **accuracy** es el porcentaje de aciertos. Parece una buena métrica, hasta que se la compara con un modelo que **no aprende nada**: el que siempre dice «no abandona».

**Predicción antes de correr:** ¿qué accuracy tiene el modelo que siempre dice «no»?

In [ ]:
y_prueba = y_test.to_numpy()
siempre_no = np.zeros(len(y_prueba), dtype=int)
rl_05 = (p_rl >= 0.5).astype(int)

comparacion = pd.DataFrame({
    "modelo": ["Siempre dice «no abandona»", "Regresión logística (umbral 0,5)"],
    "accuracy": [accuracy_score(y_prueba, siempre_no), accuracy_score(y_prueba, rl_05)],
    "recall (bajas encontradas)": [recall_score(y_prueba, siempre_no), recall_score(y_prueba, rl_05)],
})
comparacion.style.format({"accuracy": "{:.1%}", "recall (bajas encontradas)": "{:.1%}"}).hide(axis="index")

Lo que se ve: el modelo que no hace nada acierta el **85,7 %**, y la regresión logística apenas **dos puntos más**. Pero el recall muestra la diferencia real: el primero no encuentra **ninguna** baja. Con una clase tan poco frecuente, el accuracy no alcanza para elegir.

## 6 · Matriz de confusión

La matriz cruza lo que el modelo **predijo** con lo que **realmente pasó**. De ella salen todas las métricas:

- **VP** (verdadero positivo): predijo abandono y abandonó.
- **FP** (falso positivo): predijo abandono y siguió. Es un contacto «de más».
- **FN** (falso negativo): predijo que seguía y abandonó. Es un cliente perdido sin avisar.
- **VN** (verdadero negativo): predijo que seguía y siguió.

In [ ]:
def graficar_matriz(y_real, p, umbral, ax=None, titulo=None):
    pred = (p >= umbral).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_real, pred).ravel()
    ax = ax or plt.subplots(figsize=(4.8, 4))[1]
    ax.imshow([[vn, fp], [fn, vp]], cmap="Purples")
    for (i, j), (etiqueta, valor) in zip([(0, 0), (0, 1), (1, 0), (1, 1)], [("VN", vn), ("FP", fp), ("FN", fn), ("VP", vp)]):
        ax.text(j, i, f"{etiqueta}\n{valor:,}", ha="center", va="center", fontsize=13, fontweight="bold",
                color="white" if valor > max(vn, vp) * 0.5 else "#2B2B2B")
    ax.set_xticks([0, 1], ["Predijo: sigue", "Predijo: abandona"]); ax.set_yticks([0, 1], ["Real: sigue", "Real: abandonó"])
    ax.grid(False); ax.set_title(titulo or f"Umbral {umbral}")
    return vn, fp, fn, vp

vn, fp, fn, vp = graficar_matriz(y_prueba, p_rl, 0.5)
plt.show()
print(f"Precisión = VP / (VP + FP) = {vp} / {vp + fp} = {vp / (vp + fp):.1%}")
print(f"Recall    = VP / (VP + FN) = {vp} / {vp + fn} = {vp / (vp + fn):.1%}")

Lo que se ve: con el umbral de 0,5, de los clientes que el modelo marca como «va a abandonar», **7 de cada 10 abandonan de verdad** (precisión), pero el modelo **encuentra solo 27 de cada 100 bajas** (recall). Marca pocos clientes, y los que marca son confiables, pero se le escapan casi tres cuartas partes de las bajas.

## 7 · Un modelo, dos umbrales

El modelo devuelve una probabilidad; el **umbral** decide desde qué probabilidad contactamos. Bajar el umbral contacta a más clientes: sube el recall y baja la precisión.

Para decidir hace falta ponerle precio a cada error. Usamos los supuestos del caso: **contactar a un cliente que no se iba cuesta 5** (un FP) y **perder a uno que se iba cuesta 60** (un FN). Son supuestos de la clase, no datos reales.

In [ ]:
COSTO_CONTACTAR, COSTO_PERDER = 5, 60          # supuestos del caso NovaRetail

def metricas(y_real, p, umbral, c_fp=COSTO_CONTACTAR, c_fn=COSTO_PERDER):
    pred = p >= umbral
    vp = int((pred & (y_real == 1)).sum()); fp = int((pred & (y_real == 0)).sum())
    fn = int((~pred & (y_real == 1)).sum()); vn = int((~pred & (y_real == 0)).sum())
    precision = vp / (vp + fp) if vp + fp else 0.0
    recall = vp / (vp + fn) if vp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"umbral": umbral, "contactos": vp + fp, "VP": vp, "FP": fp, "FN": fn, "VN": vn,
            "precisión": precision, "recall": recall, "F1": f1, "costo": c_fp * fp + c_fn * fn}

dos_umbrales = pd.DataFrame([metricas(y_prueba, p_rl, t) for t in (0.5, 0.25)])
dos_umbrales.style.format({"precisión": "{:.1%}", "recall": "{:.1%}", "F1": "{:.3f}", "costo": "{:,.0f}"}).hide(axis="index")

Cómo leer la tabla, columna por columna: con **0,25** se contacta a unos 360 clientes en lugar de 113; el recall más que se duplica (de 27 % a 58 %); la precisión baja (de 68 % a 46 %); y el **costo total baja de 12.720 a 8.185**. Elegir el umbral no es una cuestión técnica: depende de cuánto cuesta cada error.

El gráfico siguiente muestra todo el recorrido: cada punto es un umbral posible.

In [ ]:
umbrales = np.round(np.arange(0.02, 0.91, 0.01), 2)
curva = pd.DataFrame([metricas(y_prueba, p_rl, t) for t in umbrales])

fig, ax = plt.subplots()
ax.plot(curva["umbral"], curva["precisión"], color=VIOLETA, lw=2.5, label="Precisión")
ax.plot(curva["umbral"], curva["recall"], color=VERDE, lw=2.5, label="Recall")
ax.plot(curva["umbral"], curva["F1"], color=NARANJA, lw=2.5, ls="--", label="F1")
for t in (0.5, 0.25):
    ax.axvline(t, color=GRIS, ls=":")
ax.set_xlabel("Umbral"); ax.set_ylabel("Valor"); ax.set_title("Al bajar el umbral, el recall sube y la precisión baja")
ax.legend(); plt.show()

## 8 · La capacidad manda

El equipo comercial no puede llamar a cualquier cantidad de clientes: puede atender **300 contactos por cada 2.000 clientes puntuados** (la prueba tiene exactamente 2.000). Entonces la pregunta cambia: no «qué umbral minimiza el costo», sino «a quiénes llamamos con 300 llamadas».

**Predicción antes de correr:** ¿el umbral que minimiza el costo contacta a más o a menos de 300 clientes?

In [ ]:
CAPACIDAD = 300

# 1) el umbral que minimiza el costo, sin mirar la capacidad
curva["costo"] = [metricas(y_prueba, p_rl, t)["costo"] for t in curva["umbral"]]
mejor = curva.loc[curva["costo"].idxmin()]

# 2) con capacidad: llamamos a los 300 clientes de mayor probabilidad
orden = np.argsort(-p_rl)
top = np.zeros(len(p_rl), dtype=bool); top[orden[:CAPACIDAD]] = True
umbral_capacidad = p_rl[orden[CAPACIDAD - 1]]
vp_top = int((top & (y_prueba == 1)).sum())

resumen = pd.DataFrame({
    "criterio": ["Umbral de menor costo (sin límite)", "Los 300 de mayor probabilidad"],
    "umbral": [mejor["umbral"], umbral_capacidad],
    "contactos": [int(mejor["contactos"]), CAPACIDAD],
    "bajas encontradas": [int(mejor["VP"]), vp_top],
    "costo": [mejor["costo"], COSTO_CONTACTAR * (CAPACIDAD - vp_top) + COSTO_PERDER * (int(y_prueba.sum()) - vp_top)],
})
display(resumen.style.format({"umbral": "{:.2f}", "costo": "{:,.0f}"}).hide(axis="index"))

fig, ax = plt.subplots()
ax.plot(curva["umbral"], curva["costo"], color=VIOLETA, lw=2.5)
ax.axvline(umbral_capacidad, color=NARANJA, ls="--", lw=2, label=f"Umbral con capacidad de {CAPACIDAD}: {umbral_capacidad:.2f}")
ax.scatter([mejor["umbral"]], [mejor["costo"]], color=VERDE, s=90, zorder=5, label=f"Mínimo costo: umbral {mejor['umbral']:.2f}")
ax.set_xlabel("Umbral"); ax.set_ylabel("Costo total"); ax.set_title("El umbral de menor costo supera la capacidad del equipo"); ax.legend()
plt.show()

Lo que se ve: el umbral que minimiza el costo (alrededor de 0,06) pide contactar a **más de 1.100 clientes**, casi cuatro veces lo que el equipo puede atender. Con la capacidad real, la decisión es llamar a los 300 de mayor probabilidad, lo que equivale a un umbral cercano a **0,28**. La capacidad, y no el costo, es lo que manda.

## 9 · Tres modelos, un mismo split

Comparamos tres modelos con **las mismas filas**: regresión logística, árbol de decisión (profundidad 4) y Random Forest. Para no mirar la prueba antes de tiempo, la comparación se hace con **validación cruzada dentro de entrenamiento** (5 particiones). La prueba se usa **una sola vez**, al final, para confirmar.

**Predicción antes de correr:** ¿va a ganar el ensamble por mucho?

In [ ]:
modelos = {
    "Regresión logística": LogisticRegression(max_iter=1000),
    "Árbol (profundidad 4)": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1),
}
try:                                            # XGBoost es opcional: si está instalado, se suma
    from xgboost import XGBClassifier
    modelos["XGBoost"] = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, eval_metric="logloss", random_state=42)
except ImportError:
    print("XGBoost no está instalado: la comparación sigue con los tres modelos.")

validacion = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
filas, ajustados = [], {}
for nombre, modelo in modelos.items():
    auc_cv = cross_val_score(armar_pipeline(modelo), X_train, y_train, cv=validacion, scoring="roc_auc")
    ajustados[nombre] = armar_pipeline(modelo).fit(X_train, y_train)
    p = ajustados[nombre].predict_proba(X_test)[:, 1]
    filas.append({"modelo": nombre, "AUC validación cruzada": auc_cv.mean(), "± desvío": auc_cv.std(), "AUC prueba": roc_auc_score(y_test, p)})
tabla_modelos = pd.DataFrame(filas)
display(tabla_modelos.style.format({"AUC validación cruzada": "{:.3f}", "± desvío": "{:.3f}", "AUC prueba": "{:.3f}"}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(6, 5))
for (nombre, modelo), color in zip(ajustados.items(), [VIOLETA, NARANJA, VERDE, ROJO]):
    fpr, tpr, _ = roc_curve(y_test, modelo.predict_proba(X_test)[:, 1]); ax.plot(fpr, tpr, lw=2.2, color=color, label=nombre)
ax.plot([0, 1], [0, 1], color=GRIS, ls=":"); ax.set_xlabel("Falsos positivos (tasa)"); ax.set_ylabel("Verdaderos positivos (tasa)")
ax.set_title("Curva ROC en prueba"); ax.legend(loc="lower right"); plt.show()

Cómo leer la tabla: el **AUC** mide qué tan bien el modelo ordena a los clientes (0,5 es azar, 1 es perfecto). La regresión logística obtiene **0,84** en prueba (0,83 en validación cruzada), el Random Forest **0,82**, XGBoost **0,83** (si está instalado) y el árbol **0,80**. Ningún ensamble **le gana** a la regresión logística: este problema se resuelve bien con un modelo simple. Ojo con el desvío de la validación cruzada: diferencias menores a ese desvío no son concluyentes.

## 10 · Overfitting y generalización

Un árbol puede crecer hasta **memorizar** los datos de entrenamiento. Se nota cuando mejora en entrenamiento y **empeora en prueba**. La brecha entre las dos curvas es la señal de alarma.

**Predicción antes de correr:** ¿qué le pasa al AUC de prueba cuando el árbol se hace muy profundo?

In [ ]:
profundidades = [2, 3, 4, 6, 8, 12, None]
filas = []
for d in profundidades:
    arbol = armar_pipeline(DecisionTreeClassifier(max_depth=d, random_state=42)).fit(X_train, y_train)
    filas.append({"profundidad": "sin límite" if d is None else d,
                  "AUC entrenamiento": roc_auc_score(y_train, arbol.predict_proba(X_train)[:, 1]),
                  "AUC prueba": roc_auc_score(y_test, arbol.predict_proba(X_test)[:, 1])})
sobreajuste = pd.DataFrame(filas)
sobreajuste["brecha"] = sobreajuste["AUC entrenamiento"] - sobreajuste["AUC prueba"]

fig, ax = plt.subplots()
etiquetas = sobreajuste["profundidad"].astype(str)
ax.plot(etiquetas, sobreajuste["AUC entrenamiento"], marker="o", color=VIOLETA, lw=2.5, label="Entrenamiento")
ax.plot(etiquetas, sobreajuste["AUC prueba"], marker="o", color=NARANJA, lw=2.5, label="Prueba")
ax.set_xlabel("Profundidad del árbol"); ax.set_ylabel("AUC"); ax.set_title("Más profundo: mejor en entrenamiento, peor en prueba"); ax.legend()
plt.show()
sobreajuste.style.format({"AUC entrenamiento": "{:.3f}", "AUC prueba": "{:.3f}", "brecha": "{:.3f}"}).hide(axis="index")

Lo que se ve: hasta profundidad 4 las dos curvas suben juntas. Después se separan: con profundidad 12 el árbol casi acierta todo en entrenamiento (**0,97**) y en prueba cae a **0,60**, apenas arriba del azar. El árbol sin límite llega a 1,0 en entrenamiento: memorizó. La profundidad se elige con validación, **nunca con la prueba**.

## 11 · Interpretación del modelo

Dos formas de mirar qué pesa en las predicciones: los **coeficientes** de la regresión logística (positivo: sube el riesgo de abandono; negativo: lo baja) y la **importancia de variables** del Random Forest.

In [ ]:
nombres = regresion_logistica.named_steps["preprocesamiento"].get_feature_names_out()
nombres = [n.split("__", 1)[1] for n in nombres]
coef = pd.Series(regresion_logistica.named_steps["modelo"].coef_[0], index=nombres).sort_values()
extremos = pd.concat([coef.head(6), coef.tail(6)])

importancia = pd.Series(ajustados["Random Forest"].named_steps["modelo"].feature_importances_, index=nombres).sort_values().tail(10)

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.6))
ejes[0].barh(extremos.index, extremos.values, color=[VERDE if v < 0 else NARANJA for v in extremos.values])
ejes[0].set_title("Regresión logística: coeficientes"); ejes[0].set_xlabel("Efecto sobre el riesgo (variables estandarizadas)")
ejes[1].barh(importancia.index, importancia.values, color=VIOLETA)
ejes[1].set_title("Random Forest: importancia"); ejes[1].set_xlabel("Importancia")
plt.tight_layout(); plt.show()

Cómo leer los gráficos: en la regresión logística, **naranja** sube el riesgo y **verde** lo baja. Empujan hacia el abandono: más días desde la última compra, ser del segmento Esencial, no haber respondido la encuesta y más contactos a soporte. Lo bajan: ser Premium, estar más satisfecho, comprar más y usar la app.

Una advertencia que vale para los dos gráficos: **importancia no es causalidad**. Que una variable pese en la predicción no significa que cambiarla cambie el resultado. Igual que una alta correlación no implica una relación causal.

## 12 · Calibración: ¿se puede creer la probabilidad?

Un modelo está **calibrado** si, entre los clientes a los que les da 30 % de probabilidad, abandona cerca de 30 %. Es importante cuando usamos la probabilidad para estimar costos. El gráfico agrupa a los clientes de prueba en diez grupos por probabilidad y compara lo que el modelo dijo con lo que pasó.

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 5.2))
for nombre, color in [("Regresión logística", VIOLETA), ("Random Forest", VERDE)]:
    p = ajustados[nombre].predict_proba(X_test)[:, 1]
    observada, predicha = calibration_curve(y_prueba, p, n_bins=10, strategy="quantile")
    ax.plot(predicha, observada, marker="o", lw=2, color=color, label=nombre)
ax.plot([0, 0.6], [0, 0.6], color=GRIS, ls=":", label="Calibración perfecta")
ax.set_xlabel("Probabilidad que dijo el modelo"); ax.set_ylabel("Tasa de abandono observada")
ax.set_title("La probabilidad se puede usar si sigue la diagonal"); ax.legend(); plt.show()
print(f"Error cuadrático de la regresión logística (Brier): {np.mean((p_rl - y_prueba) ** 2):.4f}")

Lo que se ve: los dos modelos siguen bien la diagonal, la regresión logística algo más. Por eso la probabilidad se puede usar para estimar costos. En un modelo mal calibrado la probabilidad **ordena** bien pero **no es creíble** como número (lo vemos en el laboratorio de errores).

## 13 · Fuga: la variable que sabe el final

Recordá `motivo_baja`: solo existe para quien ya abandonó. Probemos qué pasa si, por descuido, entra al modelo. El control es doble: miramos cuánto predice **sola** y cómo cambia el AUC al sumarla.

**Predicción antes de correr:** ¿qué AUC esperás con la variable de fuga?

In [ ]:
conmotivo = df.copy(); conmotivo["motivo_baja"] = conmotivo["motivo_baja"].fillna("ninguno")
Xf = conmotivo[NUM + CAT + ["motivo_baja"]]
Xf_train, Xf_test, yf_train, yf_test = train_test_split(Xf, conmotivo["abandono_90d"], test_size=0.25, stratify=conmotivo["abandono_90d"], random_state=42)

def preprocesamiento_fuga():
    return ColumnTransformer([
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]), NUM),
        ("cat", Pipeline([("imputar", SimpleImputer(strategy="most_frequent")), ("codificar", OneHotEncoder(handle_unknown="ignore"))]), CAT + ["motivo_baja"]),
    ])
con_fuga = Pipeline([("prep", preprocesamiento_fuga()), ("modelo", LogisticRegression(max_iter=1000))]).fit(Xf_train, yf_train)

tabla_fuga = pd.DataFrame({
    "modelo": ["Sin motivo_baja (correcto)", "Con motivo_baja (fuga)"],
    "AUC en prueba": [roc_auc_score(y_test, p_rl), roc_auc_score(yf_test, con_fuga.predict_proba(Xf_test)[:, 1])],
})
display(tabla_fuga.style.format({"AUC en prueba": "{:.3f}"}).hide(axis="index"))
print("Abandono entre quienes tienen motivo_baja:", conmotivo.loc[conmotivo["motivo_baja"] != "ninguno", "abandono_90d"].mean())

Lo que se ve: con la fuga, el AUC salta a **1,000**: el modelo «acierta todo». Un resultado perfecto en un problema real es una **señal de alarma**, no un logro. Todos los clientes con `motivo_baja` abandonaron, porque el motivo se completa **después** de la baja. En producción esa columna estaría vacía y el modelo no serviría.

## 14 · Laboratorio de errores

Los cinco errores que más cambian una conclusión. Cada uno muestra un resultado **incorrecto** y uno **controlado**. Antes de cada celda, predecí cuál va a ser el impacto.

### Error 1 · Elegir el umbral mirando la prueba

Si el umbral se elige con las mismas filas con las que después se mide, el costo que se promete es **optimista**. Lo mostramos partiendo la prueba en dos mitades al azar, 200 veces: elegimos el umbral en una mitad y medimos el costo en esa misma mitad (lo que se **promete**) y en la otra, que no se usó para elegir (lo que **pasa**). Lo correcto es elegir con datos de validación y mirar la prueba una sola vez.

In [ ]:
ts = np.round(np.arange(0.02, 0.91, 0.01), 2)

def costos_por_umbral(y_real, p):                 # costo total para cada umbral posible, de una vez
    pred = p[:, None] >= ts[None, :]
    return COSTO_CONTACTAR * (pred & (y_real[:, None] == 0)).sum(0) + COSTO_PERDER * (~pred & (y_real[:, None] == 1)).sum(0)

azar = np.random.default_rng(0)
prometido, real, elegidos = [], [], []
for _ in range(200):
    orden_azar = azar.permutation(len(y_prueba)); mitad_a, mitad_b = orden_azar[:1000], orden_azar[1000:]
    c_a = costos_por_umbral(y_prueba[mitad_a], p_rl[mitad_a]); mejor_i = c_a.argmin()
    prometido.append(c_a[mejor_i]); real.append(costos_por_umbral(y_prueba[mitad_b], p_rl[mitad_b])[mejor_i]); elegidos.append(ts[mejor_i])

print(f"Costo prometido (medido en la mitad con que se eligió): {np.mean(prometido):,.0f}")
print(f"Costo real (en la mitad que no se usó):                 {np.mean(real):,.0f}  ({np.mean(real) / np.mean(prometido) - 1:+.1%})")
print(f"El umbral elegido cambió según la mitad: de {min(elegidos):.2f} a {max(elegidos):.2f}")

Lo que se ve: el costo prometido es **alrededor de 5 % más bajo** que el que se obtiene con filas no usadas, y el umbral «óptimo» cambia según la muestra. Elegir y medir con las mismas filas premia la suerte de esa muestra. En el trabajo real, la elección se hace con validación y la prueba se mira **una sola vez**.

### Error 2 · Comparar modelos con particiones distintas

Si cada modelo se evalúa con una partición diferente, parte de la diferencia es **suerte de la partición**. Repetimos la misma regresión logística con cinco particiones distintas y miramos cuánto se mueve el AUC solo por eso.

In [ ]:
aucs = []
for semilla in (1, 2, 3, 4, 5):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, stratify=y, random_state=semilla)
    aucs.append(roc_auc_score(yb, armar_pipeline(LogisticRegression(max_iter=1000)).fit(Xa, ya).predict_proba(Xb)[:, 1]))
print("AUC de la MISMA regresión logística con cinco particiones:", [round(a, 3) for a in aucs])
print(f"Rango: {max(aucs) - min(aucs):.3f}  ->  diferencias menores a eso entre modelos pueden ser solo la partición.")

### Error 3 · Usar el accuracy como única métrica

Con el umbral de 0,25 el modelo encuentra el doble de bajas, pero el accuracy **baja**. Si se mirara solo el accuracy, se elegiría el umbral peor para el negocio.

In [ ]:
pd.DataFrame([{"umbral": t, "accuracy": accuracy_score(y_prueba, (p_rl >= t).astype(int)),
               "recall": metricas(y_prueba, p_rl, t)["recall"], "costo": metricas(y_prueba, p_rl, t)["costo"]} for t in (0.5, 0.25)]
             ).style.format({"accuracy": "{:.1%}", "recall": "{:.1%}", "costo": "{:,.0f}"}).hide(axis="index")

### Error 4 · Leer la importancia como causalidad

`satisfaccion_faltante` es una variable muy informativa: entre quienes abandonan, la encuesta falta mucho más. Pero **no es una causa**: obligar a todos a responder la encuesta no reduciría el abandono. Es una señal del cliente, no una palanca.

In [ ]:
df.groupby("satisfaccion_faltante")["abandono_90d"].agg(["mean", "count"]).rename(
    index={0: "Respondió la encuesta", 1: "No respondió"}, columns={"mean": "tasa de abandono", "count": "clientes"}
).style.format({"tasa de abandono": "{:.1%}", "clientes": "{:,.0f}"})

### Error 5 · Ignorar la calibración

Un modelo puede ordenar bien y aun así dar probabilidades **infladas**. Ponderar las clases para «ayudar» al modelo con la clase poco frecuente es un caso típico. Comparamos, entre los 300 clientes de mayor probabilidad, lo que el modelo **dice** (probabilidad promedio) con lo que **pasa** (tasa de abandono real).

In [ ]:
ponderado = armar_pipeline(RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)).fit(X_train, y_train)
filas = []
for nombre, modelo in [("Regresión logística", regresion_logistica), ("Random Forest ponderado", ponderado)]:
    p = modelo.predict_proba(X_test)[:, 1]; idx = np.argsort(-p)[:CAPACIDAD]
    filas.append({"modelo": nombre, "AUC": roc_auc_score(y_test, p), "probabilidad promedio de los 300": p[idx].mean(), "abandono real de los 300": y_prueba[idx].mean()})
pd.DataFrame(filas).style.format({"AUC": "{:.3f}", "probabilidad promedio de los 300": "{:.1%}", "abandono real de los 300": "{:.1%}"}).hide(axis="index")

Lo que se ve: el Random Forest ponderado ordena casi igual de bien (AUC parecido), pero dice que sus 300 clientes tienen una probabilidad mucho mayor que la que se observa. Si esa probabilidad se usara para estimar cuántas bajas se evitan, el resultado sería **demasiado optimista**. Ordenar bien no es lo mismo que estar calibrado.

---

## Para llevarte

- Primero **decir qué se predice** y con qué variables disponibles al decidir.
- El **accuracy engaña** cuando una clase es poco frecuente: mirá precisión, recall y costo.
- El **umbral** lo define el costo de cada error y la **capacidad** del equipo, no el modelo.
- Un modelo más complejo **no siempre gana**: compará con una línea base, con el mismo split.
- Un resultado **demasiado bueno** es una señal de fuga.
- La probabilidad se usa para decidir solo si está **calibrada**.